# 03 - Preparação do Modelo (Data Preparation)

Este notebook foca na preparação dos dados para a fase de modelagem, seguindo o framework CRISP-DM. Aqui realizaremos o tratamento de variáveis categóricas, escalonamento de atributos numéricos e a divisão dos dados para treinamento e avaliação.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

# Configurações de visualização
pd.set_option('display.max_columns', None)

## 1. Carregamento dos Dados

Carregamos o dataset processado no EDA.

In [7]:
# Carregar o dataset
data_path = os.path.join('..', 'data', 'processed', 'synthetic_energy_emissions_dataset.csv')
df = pd.read_csv(data_path)

# Derivar colunas de mes e season a partir da coluna data
df['data'] = pd.to_datetime(df['data'])
df['mes'] = df['data'].dt.month

def get_season(mes):
    if mes in [12, 1, 2]: return 'Verao'
    if mes in [3, 4, 5]: return 'Outono'
    if mes in [6, 7, 8]: return 'Inverno'
    return 'Primavera'

df['season'] = df['mes'].apply(get_season)

print(f'Formato do dataset: {df.shape}')
df.head()

Formato do dataset: (100000, 12)


,Unnamed: 0,id_empresa,data,estado,setor,porte,tipo_combustivel,consumo_kwh,fonte_energia,emissao_co2,mes,season
0,0,C907106,2025-01-09,MG,industrial,large,electric,38495.239569,eólica,408.588087,1,Verao
1,1,C685808,2025-08-03,SP,industrial,small,electric,61392.932396,térmica,51257.748638,8,Inverno
2,2,C606665,2025-05-21,SP,comercial,large,electric,1573.519047,eólica,18.393783,5,Outono
3,3,C391689,2025-07-17,DF,outros,small,electric,2552.975440,hidrelétrica,8.979976,7,Inverno
4,4,C413525,2025-03-24,SP,residencial,small,electric,170.512176,solar,8.469215,3,Outono


## 2. Seleção de Atributos (Feature Selection)

Definimos nossas variáveis independentes (X) e a variável alvo (y - `co2_emission`).

In [8]:
target = 'emissao_co2'
features = ['consumo_kwh', 'setor', 'estado', 'fonte_energia', 'mes', 'season']

X = df[features]
y = df[target]

print(f"Variavel alvo: {target}")
print(f"Atributos ({len(features)}): {features}")

Variavel alvo: emissao_co2
Atributos (6): ['consumo_kwh', 'setor', 'estado', 'fonte_energia', 'mes', 'season']


## 3. Divisão Treino e Teste

Dividimos os dados em 80% para treinamento e 20% para teste.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho Treino: {X_train.shape}")
print(f"Tamanho Teste: {X_test.shape}")

Tamanho Treino: (80000, 6)
Tamanho Teste: (20000, 6)


## 4. Pré-processamento (Pipeline)

Aplicamos as transformações necessárias de forma consistente.

In [10]:
# Identificar colunas numericas e categoricas
numeric_features = ['consumo_kwh', 'mes']
categorical_features = ['setor', 'estado', 'fonte_energia', 'season']  # era 'estacao'

# Definir transformadores
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('[OK] Pipeline de pre-processamento configurado.')


[OK] Pipeline de pre-processamento configurado.
